In [1]:
import smplx
import torch

# import open3d as o3d

model = smplx.create("/mnt/nvme/kimi/work/SMPLer-X/common/utils/human_model_files/smplx/SMPLX_NEUTRAL.npz",model_type="smplx")

betas = torch.randn([1, model.num_betas], dtype=torch.float32)
expression = torch.randn([1, model.num_expression_coeffs], dtype=torch.float32)
body_pose = torch.randn([1, 21, 3], dtype=torch.float32)
output = model(betas=betas, expression=expression, body_pose=body_pose, return_verts=True)

print(betas.shape)
print(expression.shape)
print(body_pose.shape)

vertices = output.vertices.detach().cpu().numpy().squeeze()
joints = output.joints.detach().cpu().numpy().squeeze()



torch.Size([1, 10])
torch.Size([1, 10])
torch.Size([1, 21, 3])


In [2]:
import smplx
import torch
import numpy as np

# SMPL-X 模型
model = smplx.create(
    "/mnt/nvme/kimi/work/SMPLer-X/common/utils/human_model_files/smplx/SMPLX_NEUTRAL.npz",
    model_type="smplx"
)

# 从文件加载 SMPL-X 参数
param_path = "/mnt/nvme/kimi/work/SMPLer-X/demo/results/mabaoguo/smplx/02817_0.npz"
smplx_params = np.load(param_path)
for k, v in smplx_params.items():
    print(k, v.shape)
# # 将 numpy 转为 torch.Tensor，并增加 batch 维度
# betas = torch.from_numpy(smplx_params['betas']).float().unsqueeze(0)        # (1, 10)
# expression = torch.from_numpy(smplx_params['expression']).float().unsqueeze(0)  # (1, 10)
# body_pose = torch.from_numpy(smplx_params['body_pose']).float().unsqueeze(0)     # (1, 21, 3)
# global_orient = torch.from_numpy(smplx_params['global_orient']).float().unsqueeze(0)  # (1,3)
# transl = torch.from_numpy(smplx_params['transl']).float().unsqueeze(0)          # (1,3)
# output = model(betas=betas, expression=expression, body_pose=body_pose, return_verts=True)

# vertices = output.vertices.detach().cpu().numpy().squeeze()
# joints = output.joints.detach().cpu().numpy().squeeze()


global_orient (1, 3)
body_pose (21, 3)
left_hand_pose (15, 3)
right_hand_pose (15, 3)
jaw_pose (1, 3)
leye_pose (1, 3)
reye_pose (1, 3)
betas (1, 10)
expression (1, 10)
transl (1, 3)


In [3]:
import plotly.graph_objects as go

x, y, z = vertices[:,0], vertices[:,1], vertices[:,2]
i, j, k = model.faces[:,0], model.faces[:,1], model.faces[:,2]

fig = go.Figure(data=[go.Mesh3d(
    x=x, y=y, z=z,
    i=i, j=j, k=k,
    color='lightpink',
    opacity=1.0
)])
fig.show()

In [4]:
import smplx
import torch
import numpy as np

# ---------- 1️⃣ 加载模型 ----------
model = smplx.create(
    "/mnt/nvme/kimi/work/SMPLer-X/common/utils/human_model_files/smplx/SMPLX_NEUTRAL.npz",
    model_type="smplx",
    ext='npz',
    use_pca=False
).to('cpu')

# ---------- 2️⃣ 加载 SMPLX 参数 ----------
npz_path = "/mnt/nvme/kimi/work/SMPLer-X/demo/results/Dance_origin/smplx/00085_0.npz"
params = np.load(npz_path)

# 工具函数：转换为 torch tensor
def to_tensor(x):
    return torch.tensor(x, dtype=torch.float32)

# 将 (B, n, 3) 展开成 (B, n*3)
def flatten_pose(pose):
    if pose.ndim == 3:
        return pose.reshape(pose.shape[0], -1)
    return pose

# ---------- 3️⃣ 提取并格式化 ----------
global_orient = to_tensor(params['global_orient'])          # (1, 3)
body_pose = to_tensor(params['body_pose']).unsqueeze(0)  #(1, 21, 3)
left_hand_pose = to_tensor(flatten_pose(params['left_hand_pose']))  # (1, 45)
right_hand_pose = to_tensor(flatten_pose(params['right_hand_pose']))# (1, 45)
jaw_pose = to_tensor(params['jaw_pose'])                    # (1, 3)
leye_pose = to_tensor(params['leye_pose'])                  # (1, 3)
reye_pose = to_tensor(params['reye_pose'])                  # (1, 3)
betas = to_tensor(params['betas'])                          # (1, 10)
expression = to_tensor(params['expression'])                # (1, 10)
transl = to_tensor(params['transl'])                        # (1, 3)

print(betas.shape)
print(expression.shape)
print(body_pose.shape)

# ---------- 4️⃣ 前向推理 ----------
output = model(
    betas=betas,
    expression=expression,
    body_pose=body_pose,
    global_orient=global_orient,
    left_hand_pose=left_hand_pose,
    right_hand_pose=right_hand_pose,
    jaw_pose=jaw_pose,
    leye_pose=leye_pose,
    reye_pose=reye_pose,
    transl=transl,
    return_verts=True
)

# ---------- 5️⃣ 输出 mesh ----------
vertices = output.vertices.detach().cpu().numpy().squeeze()
faces = model.faces

print("✅ SMPLX forward success")
print("vertices:", vertices.shape)
print("faces:", faces.shape)
import plotly.graph_objects as go

x, y, z = vertices[:,0], vertices[:,1], vertices[:,2]
i, j, k = model.faces[:,0], model.faces[:,1], model.faces[:,2]

fig = go.Figure(data=[go.Mesh3d(
    x=x, y=y, z=z,
    i=i, j=j, k=k,
    color='lightpink',
    opacity=1.0
)])
fig.show()


torch.Size([1, 10])
torch.Size([1, 10])
torch.Size([1, 21, 3])
✅ SMPLX forward success
vertices: (10475, 3)
faces: (20908, 3)


In [7]:
import os
import smplx
import torch
import numpy as np
import plotly.graph_objects as go
from tqdm import tqdm

# ---------- 1️⃣ 加载模型 ----------
model = smplx.create(
    "/mnt/nvme/kimi/work/SMPLer-X/common/utils/human_model_files/smplx/SMPLX_NEUTRAL.npz",
    model_type="smplx",
    ext='npz',
    use_pca=False
).to('cpu')

# ---------- 2️⃣ 工具函数 ----------
def to_tensor(x):
    return torch.tensor(x, dtype=torch.float32)

def flatten_pose(pose):
    if pose.ndim == 3:
        return pose.reshape(pose.shape[0], -1)
    return pose

def smplx_forward_from_params(params):
    global_orient = to_tensor(params['global_orient'])
    body_pose = to_tensor(params['body_pose']).unsqueeze(0)
    left_hand_pose = to_tensor(flatten_pose(params['left_hand_pose']))
    right_hand_pose = to_tensor(flatten_pose(params['right_hand_pose']))
    jaw_pose = to_tensor(params['jaw_pose'])
    leye_pose = to_tensor(params['leye_pose'])
    reye_pose = to_tensor(params['reye_pose'])
    betas = to_tensor(params['betas'])
    expression = to_tensor(params['expression'])
    transl = to_tensor(params['transl'])

    output = model(
        betas=betas,
        expression=expression,
        body_pose=body_pose,
        global_orient=global_orient,
        left_hand_pose=left_hand_pose,
        right_hand_pose=right_hand_pose,
        jaw_pose=jaw_pose,
        leye_pose=leye_pose,
        reye_pose=reye_pose,
        transl=transl,
        return_verts=True
    )
    return output.vertices.detach().cpu().numpy().squeeze()

# ---------- 3️⃣ 读取序列 ----------
folder = "/mnt/nvme/kimi/work/SMPLest-X/demo/smpl_params/Dance_origin"
npz_files = sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.endswith('.npz')])

print(f"共检测到 {len(npz_files)} 帧")

all_vertices = []
for f in tqdm(npz_files):
    params = np.load(f)
    vertices = smplx_forward_from_params(params)
    all_vertices.append(vertices)

all_vertices = np.array(all_vertices)  # (N, V, 3)
faces = model.faces
print(faces.shape)



共检测到 147 帧


100%|██████████| 147/147 [00:00<00:00, 227.43it/s]

(20908, 3)


In [6]:
# ---------- 0️⃣ 配置渲染器 ----------
import plotly.io as pio
pio.renderers.default = 'browser'

# ---------- 1️⃣ 导入依赖 ----------
import plotly.graph_objects as go
import numpy as np

# 你的 all_vertices 和 faces 已经准备好了
# ---------- 2️⃣ 创建动画 ----------


frames = []
for t in range(len(all_vertices)):
    v = all_vertices[t]
    frames.append(go.Frame(
        data=[go.Mesh3d(
            x=v[:,0], y=v[:,1], z=v[:,2],
            i=faces[:,0], j=faces[:,1], k=faces[:,2],
            color='lightpink', opacity=1.0
        )],
        name=str(t)
    ))

x0, y0, z0 = all_vertices[0].T
fig = go.Figure(
    data=[go.Mesh3d(x=x0, y=y0, z=z0,
                    i=faces[:,0], j=faces[:,1], k=faces[:,2],
                    color='lightpink', opacity=1.0)],
    frames=frames
)

# ---------- 3️⃣ 添加播放按钮 ----------
fig.update_layout(
    scene=dict(aspectmode='data',
        # xaxis=dict(range=x_range),
        # yaxis=dict(range=y_range),
        # zaxis=dict(range=z_range),
        ),
    scene_camera=dict(
        eye=dict(x=0, y=0, z=-2.5),  # y 负方向远离屏幕
        up=dict(x=0, y=-1, z=0)       # z 向上
    ),
    updatemenus=[
        dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(
                    label='▶ Play',
                    method='animate',
                    args=[None, dict(frame=dict(duration=30, redraw=True),
                                     fromcurrent=True, mode='immediate')]
                ),
                dict(
                    label='🔄 Reset Camera',
                    method='relayout',  # 这里用 relayout 改回初始 camera
                    args=[{'scene.camera': dict(
                        eye=dict(x=0, y=0, z=-2.5),
                        up=dict(x=0, y=-1, z=0)
                    )}])
            ]
        )
    ]
)

# ---------- 4️⃣ 显示 ----------
fig.show()


In [8]:
import plotly.graph_objects as go
import imageio
import os
import subprocess
import os
import smplx
import torch
import numpy as np
import plotly.graph_objects as go
from tqdm import tqdm

folder = "/mnt/nvme/kimi/work/SMPLest-X/demo/smpl_params/Dance_origin"
output_dir = "./rendered_frames/dance_origin"
# 帧所在目录
frame_dir = output_dir
video_path = os.path.join(frame_dir, "00000_video.mp4")
os.makedirs(output_dir, exist_ok=True)
fps = 30

# ---------- 1️⃣ 加载模型 ----------
model = smplx.create(
    "/mnt/nvme/kimi/work/SMPLer-X/common/utils/human_model_files/smplx/SMPLX_NEUTRAL.npz",
    model_type="smplx",
    ext='npz',
    use_pca=False
).to('cpu')

# ---------- 2️⃣ 工具函数 ----------
def to_tensor(x):
    return torch.tensor(x, dtype=torch.float32)

def flatten_pose(pose):
    if pose.ndim == 3:
        return pose.reshape(pose.shape[0], -1)
    return pose

def smplx_forward_from_params(params):
    global_orient = to_tensor(params['global_orient'])
    body_pose = to_tensor(params['body_pose']).unsqueeze(0)
    left_hand_pose = to_tensor(flatten_pose(params['left_hand_pose']))
    right_hand_pose = to_tensor(flatten_pose(params['right_hand_pose']))
    jaw_pose = to_tensor(params['jaw_pose'])
    leye_pose = to_tensor(params['leye_pose'])
    reye_pose = to_tensor(params['reye_pose'])
    betas = to_tensor(params['betas'])
    expression = to_tensor(params['expression'])
    transl = to_tensor(params['transl'])

    output = model(
        betas=betas,
        expression=expression,
        body_pose=body_pose,
        global_orient=global_orient,
        left_hand_pose=left_hand_pose,
        right_hand_pose=right_hand_pose,
        jaw_pose=jaw_pose,
        leye_pose=leye_pose,
        reye_pose=reye_pose,
        transl=transl,
        return_verts=True
    )
    return output.vertices.detach().cpu().numpy().squeeze()

# ---------- 3️⃣ 读取序列 ----------

npz_files = sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.endswith('.npz')])

print(f"共检测到 {len(npz_files)} 帧")

all_vertices = []
for f in tqdm(npz_files):
    params = np.load(f)
    vertices = smplx_forward_from_params(params)
    all_vertices.append(vertices)

all_vertices = np.array(all_vertices)  # (N, V, 3)
faces = model.faces
print(faces.shape)

frame_paths = []

for i, verts in enumerate(tqdm(all_vertices)):
    fig = go.Figure(data=[go.Mesh3d(
        x=verts[:,0],
        y=verts[:,1],
        z=verts[:,2],
        i=faces[:,0],
        j=faces[:,1],
        k=faces[:,2],
        color='lightblue',
        opacity=1.0
    )])
    
    # 设置视角、坐标轴隐藏
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X', visible=False),
            yaxis=dict(title='Y', visible=False),
            zaxis=dict(title='Z', visible=False),
            # xaxis=dict(title='X', visible=True),
            # yaxis=dict(title='Y', visible=True),
            # zaxis=dict(title='Z', visible=True),
            camera=dict(
                eye=dict(x=0, y=0, z=-2.5),
                up=dict(x=0, y=-1, z=0)
            ),
            aspectmode='data'
        ),
        margin=dict(r=0, l=0, t=0, b=0)
    )
    
    # 保存图片
    frame_path = os.path.join(output_dir, f"frame_{i:05d}.png")
    fig.write_image(frame_path, scale=2)
    frame_paths.append(frame_path)

    # FFmpeg 命令
# 假设帧名为 frame_00001.png, frame_00002.png, ...
cmd = [
    "/usr/bin/ffmpeg",
    "-y",  # 如果文件存在则覆盖
    "-framerate", str(fps),  # 帧率
    "-i", os.path.join(frame_dir, "frame_%05d.png"),  # 输入帧
    "-c:v", "libx264",  # 视频编码
    "-pix_fmt", "yuv420p",  # 兼容性
    video_path
]

# 执行命令
subprocess.run(cmd, check=True)

print(f"视频已保存: {video_path}")



100%|██████████| 147/147 [09:43<00:00,  3.97s/it]
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-li

视频已保存: ./rendered_frames/dance_origin/00000_video.mp4


frame=  147 fps=0.0 q=-1.0 Lsize=     117kB time=00:00:04.80 bitrate= 200.4kbits/s speed=5.38x    
video:115kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 2.232124%
[libx264 @ 0x587065e24700] frame I:1     Avg QP:19.93  size:  8038
[libx264 @ 0x587065e24700] frame P:37    Avg QP:22.45  size:  2025
[libx264 @ 0x587065e24700] frame B:109   Avg QP:15.48  size:   311
[libx264 @ 0x587065e24700] consecutive B-frames:  0.7%  1.4%  0.0% 98.0%
[libx264 @ 0x587065e24700] mb I  I16..4: 13.8% 80.8%  5.4%
[libx264 @ 0x587065e24700] mb P  I16..4:  0.2%  1.2%  0.2%  P16..4:  3.6%  2.1%  0.7%  0.0%  0.0%    skip:92.0%
[libx264 @ 0x587065e24700] mb B  I16..4:  0.1%  0.0%  0.0%  B16..8:  3.9%  0.2%  0.0%  direct: 0.0%  skip:95.8%  L0:51.2% L1:47.7% BI: 1.1%
[libx264 @ 0x587065e24700] 8x8 transform intra:75.0% inter:44.1%
[libx264 @ 0x587065e24700] coded y,uvDC,uvAC intra: 7.8% 9.6% 5.7% inter: 0.2% 0.1% 0.0%
[libx264 @ 0x587065e24700] i16 v,h,dc,p: 79% 10% 10%  1%
[libx2

In [ ]:
import os
import smplx
import torch
import numpy as np
import plotly.graph_objects as go
from tqdm import tqdm
import subprocess

# ---------- 配置 ----------
folder = "/mnt/nvme/kimi/work/SMPLest-X/demo/smpl_params/male_motion_test_40s"
output_dir = "./rendered_frames/male_motion_test_40s"
os.makedirs(output_dir, exist_ok=True)
frame_dir = output_dir
video_path = os.path.join(frame_dir, "00000_video.mp4")
fps = 30
scale = 1

# ---------- 1.Load SMPL-X ----------
device = 'cuda'
model = smplx.create(
    "/mnt/nvme/kimi/work/SMPLer-X/common/utils/human_model_files/smplx/SMPLX_NEUTRAL.npz",
    model_type="smplx",
    ext='npz',
    use_pca=False
).to(device)

faces = model.faces

def to_tensor(x):
    return torch.tensor(x, dtype=torch.float32, device=device)

def flatten_pose(pose):
    return pose.reshape(pose.shape[0], -1) if pose.ndim == 3 else pose

def smplx_forward_from_params(params):
    output = model(
        betas=to_tensor(params['betas']),
        expression=to_tensor(params['expression']),
        body_pose=to_tensor(params['body_pose']).unsqueeze(0),
        global_orient=to_tensor(params['global_orient']),
        left_hand_pose=to_tensor(flatten_pose(params['left_hand_pose'])),
        right_hand_pose=to_tensor(flatten_pose(params['right_hand_pose'])),
        jaw_pose=to_tensor(params['jaw_pose']),
        leye_pose=to_tensor(params['leye_pose']),
        reye_pose=to_tensor(params['reye_pose']),
        transl=to_tensor(params['transl']),
        return_verts=True
    )
    return output.vertices.detach().cpu().numpy().squeeze()

# ---------- 2️.载入所有顶点，计算固定坐标范围 ----------
npz_files = sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.endswith('.npz')])
print(f"共检测到 {len(npz_files)} 帧")

all_verts = []

print("预扫描所有顶点以生成固定坐标系…")
for f in tqdm(npz_files):
    params = np.load(f)
    verts = smplx_forward_from_params(params)
    all_verts.append(verts)

all_verts_concat = np.concatenate(all_verts, axis=0)

# 计算范围
pad = 0.3
x_min, x_max = all_verts_concat[:,0].min() - pad, all_verts_concat[:,0].max() + pad
y_min, y_max = all_verts_concat[:,1].min() - pad, all_verts_concat[:,1].max() + pad
z_min, z_max = all_verts_concat[:,2].min() - pad, all_verts_concat[:,2].max() + pad

print("固定坐标轴范围：")
print("X:", x_min, x_max)
print("Y:", y_min, y_max)
print("Z:", z_min, z_max)

# ---------- 3️.渲染函数（使用固定坐标系） ----------
def render_mesh_to_image(verts, faces, output_path):
    fig = go.Figure(data=[go.Mesh3d(
        x=verts[:,0],
        y=verts[:,1],
        z=verts[:,2],
        i=faces[:,0],
        j=faces[:,1],
        k=faces[:,2],
        color='lightblue',
        opacity=1.0
    )])

    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X', visible=True, range=[x_min, x_max]),
            yaxis=dict(title='Y', visible=True, range=[y_min, y_max]),
            zaxis=dict(title='Z', visible=True, range=[z_min, z_max]),
            camera=dict(
               eye=dict(x=0, y=0, z=-2.5),
                up=dict(x=0, y=-1, z=0)
            ),
            aspectmode='data'
        ),
        margin=dict(r=0, l=0, t=0, b=0)
    )

    fig.write_image(output_path, scale=scale)

# ---------- 4️.正式渲染每一帧 ----------
frame_paths = []
print("渲染所有帧…")
for i, verts in enumerate(tqdm(all_verts)):
    frame_path = os.path.join(output_dir, f"frame_{i:05d}.png")
    render_mesh_to_image(verts, faces, frame_path)
    frame_paths.append(frame_path)

# ---------- 5️.使用 FFmpeg 生成 MP4 ----------
cmd = [
    "/usr/bin/ffmpeg",
    "-y",
    "-framerate", str(fps),
    "-i", os.path.join(frame_dir, "frame_%05d.png"),
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    video_path
]

subprocess.run(cmd, check=True)
print(f"视频已保存: {video_path}")


共检测到 49 帧
预扫描所有顶点以生成固定坐标系…


100%|██████████| 49/49 [00:00<00:00, 330.74it/s]


固定坐标轴范围：
X: -0.7779947102069855 0.7711480736732483
Y: -1.3574100017547608 1.2380595684051514
Z: 37.9160873413086 49.312252807617185
渲染所有帧…


100%|██████████| 49/49 [02:52<00:00,  3.52s/it]
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libv

视频已保存: ./rendered_frames/dance_origin/00000_video.mp4


frame=  147 fps=0.0 q=-1.0 Lsize=     111kB time=00:00:04.80 bitrate= 188.6kbits/s speed=14.2x    
video:108kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 2.352101%
[libx264 @ 0x624d5ebadf00] frame I:1     Avg QP:17.02  size:  4665
[libx264 @ 0x624d5ebadf00] frame P:41    Avg QP:21.94  size:  1656
[libx264 @ 0x624d5ebadf00] frame B:105   Avg QP:18.96  size:   355
[libx264 @ 0x624d5ebadf00] consecutive B-frames:  1.4%  8.2%  6.1% 84.4%
[libx264 @ 0x624d5ebadf00] mb I  I16..4: 49.3% 36.6% 14.1%
[libx264 @ 0x624d5ebadf00] mb P  I16..4:  3.4%  1.7%  1.5%  P16..4:  8.6%  3.9%  1.3%  0.0%  0.0%    skip:79.6%
[libx264 @ 0x624d5ebadf00] mb B  I16..4:  0.5%  0.1%  0.0%  B16..8: 10.4%  1.5%  0.1%  direct: 0.1%  skip:87.4%  L0:52.1% L1:45.4% BI: 2.5%
[libx264 @ 0x624d5ebadf00] 8x8 transform intra:26.5% inter:33.9%
[libx264 @ 0x624d5ebadf00] coded y,uvDC,uvAC intra: 12.4% 13.7% 8.9% inter: 0.9% 0.3% 0.0%
[libx264 @ 0x624d5ebadf00] i16 v,h,dc,p: 74% 24%  2%  0%
[lib